# H05 — At home: filters and the time–frequency plane

> **References**
>
> (Orfanidis 1996), (Chatterji et al. 2004), (Torrence and Compo 1998), (Zevin et al. 2017)

## What you do here

In class we filtered the Virgo record and looked at a blip in LIGO Hanford. Here you do the same on a glitch of another family, in another detector.

**The data.**
- 32 s of LIGO Livingston strain (L1) from the first observing run, sampled at 4096 Hz.
- In the middle of the record, 16 s after the start, there is a **koi fish**: a loud glitch. The class comes from Gravity Spy.
- The file is `labs/data/Koi_Fish_fiqsKIPjAj_L1.hdf5`. It comes from the Gravitational Wave Open Science Center (GWOSC, CC BY 4.0) and it is read offline. The folder of the data is read from `labs/config.toml` with `tomllib`.

**What you do.**
1. Band-pass the record and find the glitch in time.
2. Notch the mains line at 60 Hz and look at the ASD before and after.
3. Draw the glitch in the time–frequency plane: two spectrograms and the Q transform.
4. Draw it with the Morlet wavelet and compare.

**How you work in each task.**
1. Read what we do. The functions and the numbers under `# change these` are given.
2. Write a few lines in the empty cell and compare with *What you should see*.
3. Answer the question in the task.

The same steps work on the LIGO record you downloaded for `H04`: read it with its own function and run the cells on it.

> **The symbols**
>
> - **$f_{s}$**, **$f_{N} = f_{s}/2$** — the sampling frequency and the Nyquist frequency (Hz)
> - **$b_{k}$**, **$a_{k}$** — the coefficients of a filter; **$G(z) = B(z)/A(z)$** — its transfer function; **$|G|$** — the gain
> - **$\sigma$** — the standard deviation of the filtered noise, between 1 and 10 s from the glitch
> - **$\sqrt{S}$** — the amplitude spectral density (ASD), in Hz$^{-1/2}$, from Welch's method
> - **$r$**, **$\Delta f$** — the radius of the poles of the notch and its width (Hz)
> - **$N_{w}$** — the number of samples in one window of the spectrogram
> - **$Q$** — the quality factor of a tile, its center frequency over its width
> - **$a$**, **$\omega_{0}$**, **$f_{0}$** — the scale of the wavelet (s), the Morlet parameter ($\omega_{0} = 6$) and the center frequency of the scale, $f_{0} = \omega_{0}/(2\pi a)$
> - **$\sigma_{t}$**, **$\sigma_{f}$** — the widths of a tile in time (s) and in frequency (Hz)
> - **$W(a,b)$** — the Morlet transform at the scale $a$ and the shift $b$ (s); **$|W|^{2}$** — the scalogram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import h5py
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)
sys.path.insert(0, os.path.join(here, 'labs'))

# the configuration of the laboratories: where the data are and the events we use
import tomllib
with open(os.path.join(here, 'labs', 'config.toml'), 'rb') as config_file:
    config = tomllib.load(config_file)
DATA_DIR = os.path.join(here, 'labs', config['data']['dir'])

from gwcourse_palette import P, apply_style
from gwcourse_signal import q_transform
apply_style()

# the cached data: a koi fish in LIGO Livingston; nothing is fetched
KOI = os.path.join(DATA_DIR, 'Koi_Fish_fiqsKIPjAj_L1.hdf5')


def read_glitch(path):
    """Read one glitch file of labs/data: one dataset, with its start time x0 and its time step dx.

    Parameters
    ----------
    path : str, the hdf5 file

    Returns
    -------
    (strain, times, fs) : the array x[n]; the GPS time of each sample (s); the sampling frequency (Hz)
    """
    # the only dataset of the file, named after the channel
    with h5py.File(path, 'r') as f:
        data = f[next(iter(f.keys()))]
        strain = data[()].astype(float)
        gps_start = float(data.attrs['x0'])
        dt = float(data.attrs['dx'])
    fs = 1.0 / dt
    times = gps_start + np.arange(strain.size) * dt
    record = (strain, times, fs)
    return record


def welch_asd(x, fs, seconds):
    """Return the one-sided ASD by Welch's method: Hann window, 50 percent overlap.

    Parameters
    ----------
    x       : array, the record
    fs      : float, sampling frequency (Hz)
    seconds : float, the length of a segment (s)

    Returns
    -------
    (freq, asd) : the frequencies (Hz); the amplitude spectral density (Hz^-1/2)
    """
    # the PSD of Welch's method, then its square root
    freq, psd = signal.welch(x, fs=fs, nperseg=int(seconds * fs), window='hann')
    asd = np.sqrt(psd)
    curve = (freq, asd)
    return curve


# the record, with the time t counted from the glitch, 16 s after the start
strain, times, fs = read_glitch(KOI)
t = times - times[0] - 16.0
print(f"L1: {strain.size} samples at {fs:.0f} Hz, f_N = {fs / 2:.0f} Hz, glitch at GPS {times[0] + 16.0:.3f}")
print(f"rms of the strain {strain.std():.2e}")

The workflow of this laboratory: the record, a band-pass, a notch, then two views of the time–frequency plane.

![The record of a glitch in L1, a band-pass from 40 to 300 Hz, a notch at 60 Hz, then the spectrogram and the Q transform, then the Morlet scalogram.](figures/h05home_workflow.png)

> **How to use this notebook**
>
> Run the cells in order. In every task you write a few lines in the empty cell, then compare with *What you should see*. Change one value at a time under `# change these`, run again and look at the figure. A function is defined in the task where it is first needed. Every cell runs in a few seconds.

## Task 1 · A band-pass and the glitch in time

**What we do.** In the raw strain the glitch cannot be seen: the slow wave of the seismic wall and the violin modes are much larger. A band-pass keeps the band where the detector is quiet. We design it with `signal.butter`, check that it is stable and apply it forwards and backwards with `signal.filtfilt`. Then we draw one second around the glitch in units of $\sigma$.

**What you know.**
- The design function takes the order and the corners in units of $f_{N}$. It returns $b_{k}$ and $a_{k}$.
- The filter is stable if every pole, a root of $A(z)$, is inside the unit circle: `np.roots(a)`.
- Forwards and backwards there is no delay, but the first and the last samples are corrupted. We drop `edge` seconds at each end.

**The function.** `draw_in_sigma` draws one second of a record divided by $\sigma$.

The steps of this task.

![The sampling frequency, the band and the order go into the design function, which gives the coefficients. The poles are checked, the filter runs forwards and backwards, the edges are dropped and the record is drawn in units of sigma.](figures/h05home_task1_route.png)

In [ ]:
def draw_in_sigma(t, y, sigma, title):
    """Draw the record within 0.5 s of the glitch, divided by sigma.

    Parameters
    ----------
    t     : array, time from the glitch (s)
    y     : array, the filtered record
    sigma : float, the standard deviation of the filtered noise
    title : str, the title of the plot
    """
    # one second around the glitch, in units of the noise
    near = np.abs(t) < 0.5
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t[near], y[near] / sigma, color=P['BLUE'], lw=0.7)
    ax.set(xlabel='time from the glitch (s)', ylabel=r'strain / $\sigma$', title=title)
    plt.show()


# change these
band = (40.0, 300.0)    # the two corners of the band-pass (Hz)
order = 4               # the order of the design
edge = 2.0              # seconds dropped at each end

# where we measure sigma: between 1 and 10 s from the glitch; where we look for it: within 0.5 s
quiet = (np.abs(t) > 1) & (np.abs(t) < 10)
near = np.abs(t) < 0.5
inner = np.abs(t) < 16 - edge

**Your code.**
- Design the filter: `b_band, a_band = signal.butter(order, ..., btype='band')`, with the two corners divided by $f_{N}$.
- Print the largest $|p|$ of the poles.
- `band_passed = signal.filtfilt(b_band, a_band, strain)`.
- Print the rms of `strain[inner]` and of `band_passed[inner]`.
- `sigma` is the standard deviation of `band_passed[quiet]`. Print the largest $|$`band_passed`$|$ within `near`, divided by `sigma`. Print also its time.
- Call `draw_in_sigma(t, band_passed, sigma, 'band-pass')`.
- Then run again with `band = (30.0, 300.0)`. Set it back after.

**What you should see.**
- The largest pole is just below 1: the filter is stable.
- The rms falls by a factor of about 800.
- A spike of about nine $\sigma$ at the glitch. With the lower corner at 30 Hz it is about six $\sigma$.

**Question.** Why does the glitch become smaller in units of $\sigma$ when the band starts at 30 Hz instead of 40 Hz? Look at the ASD of the record between 30 and 40 Hz: what is there?

In [ ]:
# your code here

## Task 2 · A notch on the mains line

**What we do.** The band from 40 to 300 Hz contains the line of the mains, 60 Hz in the USA. We remove it with a notch, built as in class. Two zeros on the unit circle at the angle $\theta = 2\pi f/f_{s}$ of the line and two poles at the same angle and at the radius

$$
r = 1 - \frac{\pi\,\Delta f}{f_{s}},
$$

so that the notch is $\Delta f$ wide. Then we compare the ASD around the line before and after. We also compare the glitch in units of $\sigma$.

**The functions.**
- `notch_coefficients` gives $b_{k}$, $a_{k}$ and $r$, as in class.
- `draw_line_zoom` draws two ASDs within 15 Hz of the line.

The steps of this task.

![The line and the width of the notch give the radius r. The zeros on the unit circle and the poles at radius r give B(z) and A(z). The filter runs forwards and backwards. Welch's method gives the ASD before and after.](figures/h05home_task2_route.png)

In [ ]:
def notch_coefficients(f_line, delta_f, fs):
    """The coefficients of a notch: zeros on the unit circle at the line, poles at the radius r.

    Parameters
    ----------
    f_line  : float, the frequency of the line (Hz)
    delta_f : float, the width of the notch (Hz)
    fs      : float, sampling frequency (Hz)

    Returns
    -------
    (b, a, r) : the coefficients of B(z) and A(z); the radius of the poles
    """
    # the angle of the line on the unit circle and the radius of the poles
    theta = 2 * np.pi * f_line / fs
    r = 1 - np.pi * delta_f / fs
    zeros = np.array([np.exp(1j * theta), np.exp(-1j * theta)])
    # B(z) has the zeros, A(z) has the poles r e^{+-i theta}; the coefficients are real
    b = np.real(np.poly(zeros))
    a = np.real(np.poly(r * zeros))
    notch = (b, a, r)
    return notch


def draw_line_zoom(freq, asd_before, asd_after, f_line):
    """Draw two ASDs within 15 Hz of a line.

    Parameters
    ----------
    freq       : array, the frequencies (Hz)
    asd_before : array, the ASD before the notch (Hz^-1/2)
    asd_after  : array, the ASD after the notch (Hz^-1/2)
    f_line     : float, the frequency of the line (Hz)
    """
    # 15 Hz on each side of the line, on a log vertical axis
    zoom = np.abs(freq - f_line) < 15
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.semilogy(freq[zoom], asd_before[zoom], color=P['BLUE'], lw=1.2, label='band-pass')
    ax.semilogy(freq[zoom], asd_after[zoom], color=P['SAGE_DARK'], lw=1.2, label='band-pass and notch')
    ax.set(xlabel='frequency (Hz)', ylabel=r'$\sqrt{S}$  (Hz$^{-1/2}$)')
    ax.legend(frameon=False, loc='upper left')
    plt.show()


# change these
f_line = 60.0     # the line (Hz)
delta_f = 1.0     # the width of the notch (Hz)

**Your code.**
- `b_notch, a_notch, r = notch_coefficients(f_line, delta_f, fs)`. Print $r$.
- Print the gain $|G|$ at the line, at $\Delta f/2$ from it and at 5 Hz from it: `signal.freqz(b_notch, a_notch, worN=[...], fs=fs)` returns the frequencies and $G$.
- `notched = signal.filtfilt(b_notch, a_notch, band_passed)`, with `band_passed` from task 1.
- The ASD of `band_passed[inner]` and of `notched[inner]` with `welch_asd`, segments of 4 s. Print both at the line.
- The glitch in units of $\sigma$ again, now from `notched`.
- Call `draw_line_zoom`. Then run again with `delta_f = 0.25`. Set it back after.

**What you should see.**
- $r$ is just below 1. The gain is 0 at the line, about 0.7 at half a width and 1 at 5 Hz.
- The line falls below the continuum near it. The continuum does not move.
- The glitch is a little larger in units of $\sigma$.

**Question.** We removed a line that is not part of the glitch. Why does the glitch become larger in units of $\sigma$? What changes in the record when $\Delta f$ is 0.25 Hz?

In [ ]:
# your code here

## Task 3 · The glitch in the time–frequency plane

**What we do.** The time series says when the glitch happens. The ASD of the whole record cannot say it. We draw the glitch in the time–frequency plane, three times:
1. A spectrogram with a short window, $N_{w} = 256$.
2. A spectrogram with a long window, $N_{w} = 1024$.
3. The Q transform, with $Q = 12$.

For the maps we keep a wide band, 20 to 400 Hz, below the violin modes near 500 Hz. Each row is divided by its median, so 1 is the usual level at that frequency.

**The functions.**
- `normalized_spectrogram` is the function of the class: Hann windows of $N_{w}$ samples, 64 samples apart, each row over its median.
- `q_transform` comes from the course module `gwcourse_signal`. It also divides each row by its median.
- `draw_map` draws one map on one panel.

The steps of this task.

![A band-pass from 20 to 400 Hz, the windows of N_w samples, each row divided by its median, then the Q transform at constant Q to compare.](figures/h05home_task3_route.png)

In [ ]:
def normalized_spectrogram(x, fs, n_w):
    """The spectrogram with a Hann window of n_w samples, each row divided by its median.

    Parameters
    ----------
    x   : array, the record
    fs  : float, sampling frequency (Hz)
    n_w : int, samples in one window

    Returns
    -------
    (freqs, t_mid, power) : the frequencies (Hz); the center of each window (s from the start); the power over its median
    """
    # windows that overlap by all but 64 samples, so the columns are 64 samples apart
    freqs, t_mid, sp = signal.spectrogram(x, fs=fs, window='hann', nperseg=n_w, noverlap=n_w - 64)
    power = sp / np.median(sp, axis=1, keepdims=True)
    result = (freqs, t_mid, power)
    return result


def draw_map(ax, t_map, f_map, power, title):
    """Draw one time-frequency map on a log frequency axis, from 20 to 400 Hz.

    Parameters
    ----------
    ax           : matplotlib axes
    t_map, f_map : arrays, the times (s from the glitch) and the frequencies (Hz)
    power        : 2D array (len(f_map), len(t_map)), the map over its median
    title        : str, the title of the panel

    Returns
    -------
    mesh : the drawn map, for the color bar
    """
    # the same color scale on every panel
    mesh = ax.pcolormesh(t_map, f_map, power, vmin=0, vmax=25, shading='auto', cmap='magma')
    ax.set(yscale='log', ylim=(20, 400), xlabel='time from the glitch (s)', title=title)
    ax.set_yticks([20, 50, 100, 200, 400], labels=['20', '50', '100', '200', '400'])
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    return mesh


# change these
wide_band = (20.0, 400.0)    # the band kept for the maps (Hz)
windows = (256, 1024)        # N_w of the two spectrograms
q = 12.0                     # Q of the Q transform
half = 0.5                   # we draw the maps from -half to +half (s)

**Your code.**
- `wide`: the record band-passed from 20 to 400 Hz, forwards and backwards, order 4.
- For each $N_{w}$: call `normalized_spectrogram(wide, fs, n_w)`. Subtract 16 s from its times. Keep the columns within `half` of the glitch and the rows from 20 to 400 Hz. Print the length of the window in ms, the step of the bins in Hz and where the map is largest (time and frequency).
- The Q transform: `q_transform(wide / wide.std(), fs, t, 0.0, half=half, q=q, frange=wide_band)`. The fourth argument is the time of the center of the map, 0 s. It returns the times, the frequencies and the energy. Print where it is largest.
- One figure with three panels, `fig, ax = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)`, one `draw_map` on each. Add `fig.colorbar(mesh, ax=ax)`.

**What you should see.**
- The koi fish is a short event, centered at 0, with most of its energy below about 200 Hz.
- With $N_{w} = 256$ it is a thin column, with coarse steps in frequency. With $N_{w} = 1024$ the steps are finer, but the glitch is wider in time.
- In the Q transform the shape is a head at low frequency and a narrow tail to higher frequency, the shape that gives the family its name.

**Question.** Which map tells best when the glitch starts and which tells best at which frequencies it is? Why does the Q transform show the low-frequency part longer in time than the high-frequency part?

In [ ]:
# your code here

## Task 4 · The Morlet scalogram

**What we do.** We draw the same glitch with the Morlet wavelet, as in class. For each center frequency $f_{0}$ the scale is $a = \omega_{0}/(2\pi f_{0})$. The scalogram $|W(a,b)|^{2}$ is divided by its median, row by row. Then we compare it with the Q transform of task 3.

**What you know.** For the Morlet wavelet $Q = \sqrt{2}\,\omega_{0}$ at every scale, with

$$
\sigma_{t} = \frac{a}{\sqrt{2}}, \qquad \sigma_{f} = \frac{1}{2\sqrt{2}\,\pi a}.
$$

**The function.** `morlet_scalogram` is the function of the class.

The steps of this task.

![The center frequencies give the scales. The data are correlated with the Morlet wavelet at each scale and the squared modulus, over its median, is the scalogram.](figures/h05home_task4_route.png)

In [ ]:
def morlet_scalogram(x, fs, freqs, w0=6.0):
    """The Morlet scalogram |W(a,b)|^2, each row divided by its median.

    Parameters
    ----------
    x     : array, the record
    fs    : float, sampling frequency (Hz)
    freqs : array, the center frequencies f_0 of the rows (Hz)
    w0    : float, the Morlet parameter omega_0

    Returns
    -------
    power : array (len(freqs), len(x)), the scalogram over its median, row by row
    """
    power = np.empty((freqs.size, x.size))
    for row, f0 in enumerate(freqs):
        # the scale of this row and the wavelet on +-5 a, where the Gaussian has fallen to 4e-6
        a = w0 / (2 * np.pi * f0)
        tw = np.arange(-5 * a, 5 * a, 1 / fs)
        psi = np.pi ** -0.25 * np.exp(1j * w0 * tw / a) * np.exp(-(tw / a) ** 2 / 2) / np.sqrt(a)
        # W(a, b) for every shift b: the correlation of the data with the wavelet, times dt
        w_ab = signal.correlate(x, psi, mode='same', method='fft') / fs
        power[row] = np.abs(w_ab) ** 2
    power = power / np.median(power, axis=1, keepdims=True)
    return power


# change these
w0 = 6.0                                  # the Morlet parameter
f_rows = np.geomspace(20.0, 400.0, 80)    # the center frequencies of the rows (Hz)
seg = np.abs(t) < 4                       # the 8 s of the record that we transform

**Your code.**
- `scal = morlet_scalogram(wide[seg], fs, f_rows, w0)`, with `wide` from task 3.
- Keep the columns within `half` of the glitch. Print where the scalogram is largest: time and frequency.
- At that frequency, compute $a$, $\sigma_{t}$ and $\sigma_{f}$ and print them. Do the same at 300 Hz.
- One figure with two panels: the scalogram and the Q transform of task 3 (`t_q`, `f_q`, `energy`), with `draw_map`.

**What you should see.**
- The scalogram has the same shape as the Q transform.
- At the frequency of the maximum the tile is a few tens of ms long. At 300 Hz it is a few ms long.

**Question.** The Q transform of task 3 has $Q = 12$ and the Morlet wavelet has $Q = \sqrt{2}\,\omega_{0} \approx 8.5$. Which of the two maps should be sharper in time at the same frequency? Does the figure agree?

In [ ]:
# your code here

## What to hand in

**For each task.** The answer to the question, in two or three sentences. Add the figure, the values you changed and what you saw.

1. **Task 1.** The largest pole, the rms before and after and the glitch in units of $\sigma$ for the two bands. What is between 30 and 40 Hz.
2. **Task 2.** $r$, the gain at three frequencies and the ASD at 60 Hz before and after. Why the glitch grows in units of $\sigma$.
3. **Task 3.** The three maps on one color scale. Which map says best when and which says best at which frequencies.
4. **Task 4.** The scalogram beside the Q transform, with $\sigma_{t}$ and $\sigma_{f}$ at two frequencies.

> **Where this goes next**
>
> - A filter is fixed by its coefficients. Check its poles and its gain before you use it.
> - In the raw strain a glitch cannot be seen: the seismic wall and the lines are much larger. A band-pass and a notch show it.
> - The spectrogram has one tile shape for the whole plane. The Q transform and the Morlet wavelet change the shape with the frequency.

## Software

`numpy`, `scipy.signal` (`butter`, `filtfilt`, `freqz`, `welch`, `spectrogram`, `correlate`) and `matplotlib` (Virtanen et al. 2020). The notch follows (Orfanidis 1996). The Q transform is `q_transform` of the course module `gwcourse_signal`, after (Chatterji et al. 2004). The Morlet wavelet with $\omega_{0} = 6$ follows (Torrence and Compo 1998). The data come from GWOSC (LIGO Scientific Collaboration and Virgo Collaboration 2021); the class of the glitch comes from Gravity Spy (Zevin et al. 2017).

**References**

- Orfanidis (1996). *Introduction to Signal Processing*.
- Chatterji et al. (2004). *Multiresolution techniques for the detection of gravitational-wave bursts*.
- Torrence and Compo (1998). *A Practical Guide to Wavelet Analysis*.
- Zevin et al. (2017). *Gravity Spy: integrating advanced LIGO detector characterization, machine learning, and citizen science*.
- Virtanen et al. (2020). *SciPy 1.0: fundamental algorithms for scientific computing in Python*.
- LIGO Scientific Collaboration and Virgo Collaboration (2021). *Open data from the first and second observing runs of Advanced LIGO and Advanced Virgo*.